# A1 · Reducción raw (esorex)

**Spec:** [`docs/spec_A1_codex_raw_reduction.md`](../docs/spec_A1_codex_raw_reduction.md)  |  **Bloque:** A · Reducción  |  **Run por defecto:** `ROXs12b_realigned`

Reduce los raw MUSE con esorex y alinea las exposiciones hasta `cube_telcorr.fits`.

| | |
|---|---|
| **Entrada** | Raw MUSE + calibraciones |
| **Salida (QC/productos)** | `cube_telcorr.fits`, `stages/stage00r_qc.json` |
| **Consume aguas abajo** | Todo el bloque B |


## Cómo ejecutar de forma independiente

> ⚠️ **Etapa no re-ejecutable desde raw en este repo.** En la poda WP-10 se borraron los intermedios regenerables (`muse_scibasic`, `muse_scipost`, …). Se conservaron los productos finales y todo el QC. Este notebook **audita** el producto/QC existente y documenta el comando histórico.

Comando histórico (referencia, requiere los raw + `esorex`):

```bash
conda activate MUSE
bash scripts/reduce_raw.sh
```


## Coste de ejecución (esorex)

> ⏱️ **Referencia real** medida en esta máquina (esorex 3.13.10 / MUSE 2.10.16, dataset NFM-AO de ROXs 12: **7 exposiciones × 24 IFUs = 168 pixtables**).

| Receta | Tiempo | Escala con |
|---|---:|---|
| bias | 33 min | calibración (~fijo) |
| flat | 52 min | calibración |
| wavecal | 51 min | calibración |
| lsf | 50 min | calibración |
| scibasic (std) | 4.5 min | 1× |
| standard | 1.6 min | 1× |
| scibasic (object) | 24 min | **N_exp** |
| scipost | 73 min | **N_exp** |
| **Total (7 exp)** | **≈ 289 min (~4.8 h)** | |

**Fórmula para datos nuevos** (mismo instrumento/máquina):

```
T(min) ≈ T_cal + T_std + N_exp·(t_scibasic + t_scipost) + T_combine
```

con constantes medidas aquí:

- `T_cal ≈ 186 min` = bias+flat+wavecal+lsf. **Una vez por noche/modo**; `0` si reutilizas los master calibrations.
- `T_std ≈ 6 min` = scibasic_std + standard (una vez).
- `t_scibasic ≈ 3.4 min/exp`, `t_scipost ≈ 10.5 min/exp` (24 IFUs c/u; plan B = scipost por exposición).
- `T_combine ≈ 5 min` = muse_exp_align + muse_exp_combine (plan B, offsets manuales).

**Nota:** scibasic/scipost paralelizan sobre los 24 IFUs (OpenMP) → el tiempo escala aprox. inverso al nº de núcleos; `cores_factor` ajusta ese factor respecto a esta máquina base (=1.0). La calibración domina: reutilizar masters recorta ~3 h.


In [ ]:
def estimate_esorex_runtime(n_exp, reuse_calibrations=False, cores_factor=1.0):
    """Estima el wall-time de la reducción esorex (min), calibrada en la
    máquina de referencia (7 exp NFM-AO ~= 289 min). Ver tabla de arriba."""
    T_cal = 0.0 if reuse_calibrations else 186.0  # bias+flat+wavecal+lsf
    T_std = 6.0                                    # scibasic_std + standard
    t_scibasic, t_scipost = 3.4, 10.5             # min por exposición (24 IFU)
    T_combine = 5.0                                # exp_align + exp_combine
    return (T_cal + T_std + n_exp * (t_scibasic + t_scipost) + T_combine) / cores_factor

for n in (1, 3, 7, 10):
    m = estimate_esorex_runtime(n)
    print(f'{n:2d} exp  ->  {m:5.0f} min  (~{m/60:.1f} h)')
print('7 exp reutilizando masters ->',
      f'{estimate_esorex_runtime(7, reuse_calibrations=True):.0f} min')


In [ ]:
import os, sys
# Añade notebooks/ (para _nbcommon) y la RAÍZ del repo (para importar musepipe),
# funcione el cwd en notebooks/ o en la raíz del repo.
_here = os.getcwd()
if os.path.basename(_here) != 'notebooks' and os.path.isdir(os.path.join(_here, 'notebooks')):
    _here = os.path.join(_here, 'notebooks')
for _p in (_here, os.path.dirname(_here)):
    if _p not in sys.path:
        sys.path.insert(0, _p)
import _nbcommon as nb
_root = str(nb.project_root())
if _root not in sys.path:
    sys.path.insert(0, _root)   # asegura 'import musepipe'
RUN_ID = nb.resolve_run_id(None)
print('run  =', RUN_ID)
print('root =', _root)
print('dir  =', nb.run_dir(RUN_ID))


## Auditar

Etapa de solo-auditoría: se carga el producto/QC más abajo.


## QC / resultados


In [ ]:
qc = nb.load_qc('stages/stage00r_qc.json', RUN_ID)
nb.show(qc, keys=['shape', 'sha', 'offset', 'esorex', 'muse'], title='A1')


## Verificaciones (V1–V6): qué comprueban y qué respondieron

El QC de A1 (`stage00r_qc.json`) no re-reduce: **verifica** que el cubo entregado es sano y trazable. Cada check tiene un significado concreto:

| Check | Qué comprueba | Resultado | Significado |
|---|---|---|---|
| **V1** STAT | La extensión STAT (varianza) existe, es positiva y con pocos NaN (excluyendo canales láser AO y spaxels de borde) | **ok** (NaN 0.24%, 216 canales láser y 6972 spaxels de borde excluidos) | El cubo trae su mapa de varianza y no está corrupto → base para toda la propagación de error aguas abajo |
| **V2** estándar | Continuo del estándar vs su curva de respuesta | **unavailable** | No hay curva de respuesta del estándar para esta reducción → no se pudo cerrar la validación de flujo relativa aquí (se cierra por otra vía en A4/M3 vs Gaia) |
| **V3** WCS | `CRVAL3` y paso espectral correctos | **ok** | Solución de longitud de onda y WCS sanos → los λ del cubo son fiables |
| **V4** vs ADP | Correlación de la imagen luz-blanca del cubo propio con la del ADP de ESO | **ok** (corr = 0.9994, shift entero (0,0)) | El cubo auto-reducido reproduce la morfología del ADP oficial → validación cruzada independiente de la reducción |
| **V5** espectro estelar | Razón del espectro de la estrella entre cubo y ADP | **unavailable** | Los dos cubos ponen la estrella en píxeles distintos; hace falta registro por-cubo → no comparable con la API punto-único |
| **V6** máscara de cielo | La máscara de cielo de scipost es limpia | **unavailable** | scipost no exporta la máscara como producto 2D verificable → no auditable aquí |

V1/V3/V4 pasan; V2/V5/V6 son **lagunas de proveniencia documentadas** (no fallos físicos). Por eso el semáforo A1 = **yellow**. La celda de abajo los imprime en vivo.


In [ ]:
q = nb.load_qc('stages/stage00r_qc.json', RUN_ID)
labels = {
    'v1_stat_present':      'V1 · STAT presente y sano',
    'v2_std_residual_rms':  'V2 · Residuo del estándar (respuesta de flujo)',
    'v3_wcs_ok':            'V3 · WCS / eje espectral',
    'v4_adp_whitelight_corr':'V4 · Correlación luz-blanca vs ADP',
    'v5_adp_star_spec_ratio':'V5 · Razón de espectro estelar vs ADP',
    'v6_sky_mask_clean':    'V6 · Máscara de cielo limpia',
}
ver = q.get('verification', {})
for k, lab in labels.items():
    v = ver.get(k, {})
    res = 'ok' if v.get('ok') else v.get('status', '?')
    print(f'{lab}\n   -> {res}\n   {v.get("message", "")}\n')


## Decisiones y notas
- **Alineación por plan B (OFFSET_LIST manual)**, no `exp_align` — daba offsets espurios de hasta 3.305" (cross-match de speckles NFM); el manual desde el centroide de la primaria da máx 0.62". El cubo realineado ≡ ADP a través del bloque B.
- Provenance QC = **AMARILLO**: V1/V3/V4 pasan; V2/V5/V6 = `unavailable` (lagunas documentadas, no fallos). Ver tabla de verificaciones arriba.
- **Estado A-block (actualizado 2026-07-10):** los 6 blockers duros están **CERRADOS** → F1 realineado = `yellow`, **0 bloqueantes**. Quedan 4 `open_issues` NO bloqueantes (V2/V5/V6, agrupación de calibraciones BIAS, molecfit no convergió→STD_TELLURIC). El paquete ya **no bloquea por A**; la validez para paper es juicio científico con esos caveats declarados. **Supera la directiva absoluta del 2026-07-07.**


## Checks


In [ ]:
print('open_issues A1 (no bloqueantes):')
for i, s in enumerate(qc.get('open_issues', []), 1):
    print(f'  {i}. {s}')


## Conclusión (registrada)

**A1: `cube_telcorr.fits` reducido y alineado; semáforo A1 = `yellow` (no bloqueante).**

- **Fecha:** reducción 2026-07-08 (esorex 3.13.10 / MUSE 2.10.16); provenance QC escrito 2026-07-09.
- **Datos:** 7 exposiciones NFM-AO (OB 3444577, Prog 109.23B7.002, MUSE.2022-09-01T00:36–02:03).
- **Alineación:** plan B, OFFSET_LIST manual desde el centroide de la primaria (máx 0.62"), porque `muse_exp_align` dio offsets espurios de hasta 3.305".
- **Verificaciones:** V1/V3/V4 pass; V2/V5/V6 `unavailable` (documentadas).
- **Cubo:** 3681×330×338, sha256 `9fff16b7…`; corr luz-blanca vs ADP = 0.9994.
- **A-block:** 6 blockers duros cerrados (F1 yellow, 0 bloqueantes); 4 caveats no bloqueantes documentados. Nada es aún paper-final sin declarar esos caveats.
